# Detection pipeline

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs. (Real sessions need their full artifacts — e.g. opto-pattern images for opto sessions — or parsing will raise.)

In [ ]:
import os
import polars as pl

from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.core.hub import Hub
from piepy.tasks.wheel_detection.wheelDetectionSession import WheelDetectionSession

from piepy.viz.plots import psychometric
from piepy.viz.trial.wheel_detection import trial_snapshot

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Parse a single session
Builds the Session, parses each run, and stacks them onto one session clock. Wrapped defensively so a real-data hiccup prints a clear error instead of aborting.

In [ ]:
# sess = WheelDetectionSession("230106_KC144_detect__no_cam_KC")
# data = sess.analyze(load_flag=True)

sess = WheelDetectionSession("240810_KC150_detect__no_cam_KC")
data = sess.analyze(load_flag=False)

In [ ]:
for c in ["session_uid", "run_uid", "run_no", "paradigm"]:
    assert c in data.columns

In [ ]:
data.select(["outcome","t_trialstart","t_trialend"])

## Plotting a trial snapshot

In [ ]:
trial_snapshot(data,trial_no=18)

## Plotting from a single Run/Session

### Plotting from a dataframe

In [ ]:
df_side = data.filter(pl.col("stim_side")!="ipsi")

pr = psychometric(data=data,
                  x="contrast",
                  outcome="outcome",
                  success="hit",
                  fit_curve=True,
                  model="naka-rushton",
                  palette=("#DD7703","#0164E5"),
                  label='Opto ')

#### You can pass ```kwargs``` that override the visual properties of the plots

This requires a bit of knowledge of the composition of the plot. Psychometric plot is made from an ```errorbar``` and ```line``` plot. You can target the style of these components by prefixing your ```kwargs``` with their names, e.g. ```errorbar_linewidth```, ```line_linestyle```.

In [ ]:
pr = psychometric(data=df,
                  x="contrast",
                  outcome="outcome",
                  compare="opto",
                  success="hit",
                  fit_curve=True,
                  palette=("#DD7703","#0164E5"),
                  label='Opto ',
                  errorbar_linewidth=9,
                  line_linestyle=":"
                  )

#### The returned ```PlotResult``` object has the data, statistical tests(if applicable) and the (fig,ax) tuple

In [ ]:
pr

### Plotting with accessor

You can pass a ```filterer``` argument to filter the data

In [ ]:
pr = sess.viz.psychometric(filterer={"stim_side":["catch","contra"]},
                           x="contrast",
                           outcome="outcome",
                           compare="opto",
                           success="hit",
                           fit_curve=True,
                           palette=("#DD7703","#0164E5"),
                           label='Opto ')

In [ ]:
pr = sess.viz.reaction_time_cloud(filterer={"outcome":"hit"},
                                  x="contrast",
                                  value="reaction_time",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=10,
                                  label='Opto',
                                  dodge_width=0.2,
                                  violin_widths=0.05,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=0.05,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF")

In [ ]:
pr = sess.viz.reaction_time_dist(filterer={"outcome":"hit",
                                           "contrast":0.5},
                                  value="reaction_time",
                                  bin_width=5,
                                  alpha=0.8)

## Cohort across many sessions (`Hub`)
`Hub` runs each session in parallel and stacks them into one cohort table. It already isolates per-session failures (a bad session warns and is skipped).

In [ ]:
sessions = ['221108_KC142_detect_opto120_HVA__no_cam_KC',
            '230228_KC143_detect_opto120_HVA__no_cam_KC',
            '230228_KC144_detect_opto120_HVA__no_cam_KC',
            '240514_KC149_detect_opto120_HVA__no_cam_KC',
            '240625_KC147_detect_opto120_HVA__no_cam_KC',
            '240627_KC148_detect_opto120_HVA__1P_KC',
            '240817_KC151_detect_opto120_HVA__no_cam_KC',
            '240828_KC152_detect_opto120_HVA__no_cam_KC']

In [ ]:
cohort = None
if sessions:
    try:
        hub = Hub("wheel_detection")
        hub.initialize([os.path.basename(s) for s in sessions], load_sessions=False)
        cohort = hub.data
        print('cohort:', None if cohort is None else cohort.shape)
    except Exception as e:
        print("Hub gather failed:", type(e).__name__, e)

### When looking at multiple sessions, the ```average_over``` argument is defaulted to ```animalid``` to average over animals, if ```None``` is passed all the trials will be pooled

In [ ]:
pr = hub.viz.psychometric(filterer={"stim_side":["catch","contra"]},
                          x="contrast",
                          outcome="outcome",
                          average_over="animalid",
                          compare="opto",
                          success="hit",
                          fit_curve=True,
                          model="weibull",
                          palette=("#DD7703","#0164E5"),
                          label='Opto ')

In [ ]:
pr = hub.viz.reaction_time_cloud(filterer={"outcome":"hit",
                                            "contrast":[0.125,0.5]},
                                  x="signed_contrast",
                                  value="reaction_time",
                                  compare="opto",
                                  average_over="animalid",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=50,
                                  label='Opto',
                                  dodge_width=0.3,
                                  violin_widths=0.2,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=0.1,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF")

In [ ]:
pr = hub.viz.reaction_time_dist(filterer={"outcome":"hit",
                                           "contrast":0.5},
                                  value="reaction_time",
                                  comparing="opto",
                                  average_over=None,
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=5,
                                  label='Opto ',
                                  alpha=0.8)